# Spark X2.6 4B Loop72 validation and benchmark runs

This notebook separates runtime checks, smoke tasks, inference timing, complete dataset runs, and actual Oh My Pi coding tests. The fast highway and trained mHC are distinct variants; both execute 72 visits. Run from the reviewed checkout. Full dataset adapters currently cover GraphWalks BFS, MRCR 8 needle, and LongBench v2. Other suites require their upstream harnesses.

See `docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md` for the training sequence and release criteria.


In [ ]:
import json
from pathlib import Path
import torch
import sparkx_full_benchmark as full
import sparkx_loop72_dlsa_benchmarks as bench
import load_inference_v03 as loader

RUN_DIR = Path('runs/loop72_validation')
RUN_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT = full.preflight()
(RUN_DIR / 'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2))
print(json.dumps(PREFLIGHT, indent=2))
# Preflight checks prerequisites; it is not a model benchmark.


In [ ]:
# Use the exported checkpoint directory after mHC training.
CHECKPOINT_DIR = None
DLSA_CHECKPOINT = None
MODEL_REVISION = full.MODEL_REVISION
RECURRENCE_MODE = 'learned_mhc' if CHECKPOINT_DIR else 'depth_matched_highway'
model, tokenizer = loader.load_sparkx_loop72_lsa(
    repo_or_dir=CHECKPOINT_DIR or 'XHToken/Spark-X2.6-4B',
    revision=MODEL_REVISION, dlsa_checkpoint=DLSA_CHECKPOINT,
    recurrence_mode=RECURRENCE_MODE,
)
manifest = model.execution_manifest()
assert manifest['total_virtual_layers'] == 72
assert manifest['global_visits'] == 18 and manifest['local_visits'] == 54
assert manifest['engram_constructed'] is False
print(json.dumps(manifest, indent=2))
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
# Explicit smoke tasks test plumbing; these are not official scores.
SMOKE_REPORT = bench.evaluate_selected_benchmark_tasks(
    model, tokenizer, max_new_tokens=2048,
    include_hf_streamed=False, include_smoke=True, task_mode='diagnostic',
)
(RUN_DIR / 'smoke.json').write_text(json.dumps(SMOKE_REPORT, indent=2))
print(json.dumps(SMOKE_REPORT, indent=2))


In [ ]:
# Fresh caches, repeated measurements, actual generated counts.
SPEED_REPORT = bench.benchmark_loop72_decode_speed(
    model, tokenizer, prompt_lengths=(512, 2048, 8192, 32768),
    gen_tokens=512, warmups=1, repeats=3,
)
(RUN_DIR / 'speed.json').write_text(json.dumps(SPEED_REPORT, indent=2))
print(json.dumps(SPEED_REPORT, indent=2))


In [ ]:
# Full means every eligible row of each pinned dataset, without a sample cap.
RUN_FULL_DATASETS = False
if RUN_FULL_DATASETS:
    if not DLSA_CHECKPOINT:
        raise RuntimeError('Load a validated D-LSA checkpoint before long-context quality evaluation.')
    FULL_REPORT = full.run_full_benchmarks(
        model, tokenizer, RUN_DIR / 'full', suites=None,
        max_new_tokens=32768, limit=None,
    )
else:
    FULL_REPORT = {'status': 'not_run', 'complete': False, 'reason': 'RUN_FULL_DATASETS is False'}
(RUN_DIR / 'full_status.json').write_text(json.dumps(FULL_REPORT, indent=2))
print(json.dumps(FULL_REPORT, indent=2))


In [ ]:
# Capacity and throughput only; repeated text does not prove retrieval quality.
RUN_CONTEXT_SCALING = False
if RUN_CONTEXT_SCALING:
    CONTEXT_REPORT = bench.run_full_length_1m_context_benchmark(
        model=model, tokenizer=tokenizer,
        horizons=(32768, 131072, 262144, 524288, 1048576),
        max_live_tokens=131072, gen_tokens=512,
    )
else:
    CONTEXT_REPORT = {'status': 'not_run', 'reason': 'RUN_CONTEXT_SCALING is False'}
(RUN_DIR / 'context_status.json').write_text(json.dumps(CONTEXT_REPORT, indent=2))
print(json.dumps(CONTEXT_REPORT, indent=2))


In [ ]:
# Generate a project example; preserve any existing personal models.yml.
from sparkx_omp_server import generate_omp_models_yaml
OMP_CONFIG = generate_omp_models_yaml(
    output_path=str(RUN_DIR / 'omp_models.yml'), port=8000,
    model_id='spark-x2.6-4b-loop72-dlsa', context_window=32768,
)
print(OMP_CONFIG)
RUN_SERVER = False
if RUN_SERVER:
    omp_server = bench.start_loop72_openai_compatible_server(model, tokenizer, port=8000, background=True)
# In the same host/network namespace, run: python tools/omp_eval.py --help
# Use its actual OMP coding fixture; inspect raw events and the external test result.
# Colab localhost and laptop localhost are different; use a controlled tunnel if needed.


## Training and release sequence

Train mHC with native attention first and validate checkpoint persistence and train/eval parity. Fit per-visit D-LSA selectors against that frozen checkpoint, using calibration data separate from evaluation. Quantized cache and extreme context extension follow the BF16 dense/sparse acceptance gates. The detailed recipe and upstream benchmark requirements are in `docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md`.
